# Generate HNSW Embeddings in Contiguous Shards (Kaggle GPU)

Shard the corpus into contiguous chunks to stay within Kaggle memory limits. Run this notebook multiple times, changing `SHARD_IDX` (0..NUM_SHARDS-1). Each run processes a contiguous slice of `collection.tsv` with no overlap. Query embeddings are generated once.

**Inputs (attach under Kaggle Data):**
- `data/collection/collection.tsv`
- `data/queries/queries.all.tsv`

**Outputs (per run):**
- `/kaggle/working/artifacts/hnsw_embeddings/doc_embeddings_part{SHARD_IDX}.h5`
- `/kaggle/working/artifacts/hnsw_embeddings/query_embeddings.h5` (run once)

Download shard files and merge/iterate locally to compute dense scores and build tiered HNSW indexes.

In [ ]:
# Install minimal dependencies
!pip install -q sentence-transformers h5py

# Clone the repository (for configs if needed)
import os, sys, subprocess
REPO_URL = "https://github.com/timothycao/search-systems.git"
REPO_DIR = "/kaggle/working/search-systems"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

In [ ]:
import h5py
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Paths (adjust dataset name if different)
DOCS_TSV = Path("/kaggle/input/tiering-artifacts/data/collection/collection.tsv")
QUERIES_TSV = Path("/kaggle/input/tiering-artifacts/data/queries/queries.all.tsv")

OUT_DIR = Path("/kaggle/working/artifacts/hnsw_embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Shard settings (contiguous ranges)
SHARD_IDX = 0       # set 0..NUM_SHARDS-1 per run
NUM_SHARDS = 5      # total shards
BATCH_SIZE = 512
BLOCK_SIZE = 50000  # docs per block for streaming/encoding
MODEL_NAME = "sentence-transformers/msmarco-bert-base-dot-v5"
str_dtype = h5py.string_dtype(encoding="utf-8")

model = SentenceTransformer(MODEL_NAME)

def stream_tsv_contiguous(path: Path, start_line: int, max_lines: int, block_size: int):
    ids, texts = [], []
    seen = 0
    emitted = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            if seen < start_line:
                seen += 1
                continue
            if emitted >= max_lines:
                break
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) != 2:
                seen += 1
                continue
            ids.append(parts[0]); texts.append(parts[1]); seen += 1; emitted += 1
            if len(ids) >= block_size:
                yield ids, texts
                ids, texts = [], []
    if ids:
        yield ids, texts

def encode_docs_shard(tsv_path: Path, out_path: Path, shard_idx: int, num_shards: int):
    # Compute shard boundaries
    total_lines = sum(1 for _ in tsv_path.open("r", encoding="utf-8"))
    shard_size = (total_lines + num_shards - 1) // num_shards
    start_line = shard_idx * shard_size
    max_lines = min(shard_size, total_lines - start_line)

    offset = 0
    with h5py.File(out_path, "w") as h5f:
        ids_ds = h5f.create_dataset("id", shape=(0,), maxshape=(None,), dtype=str_dtype, compression="gzip")
        emb_ds = None
        for ids, texts in stream_tsv_contiguous(tsv_path, start_line, max_lines, BLOCK_SIZE):
            emb = model.encode(
                texts,
                batch_size=BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=False,
            ).astype(np.float32)
            ids_arr = np.array(ids, dtype=object)
            if emb_ds is None:
                emb_ds = h5f.create_dataset(
                    "embedding",
                    shape=(0, emb.shape[1]),
                    maxshape=(None, emb.shape[1]),
                    dtype="float32",
                    compression="gzip",
                )
            new_size = offset + len(ids_arr)
            ids_ds.resize((new_size,))
            emb_ds.resize((new_size, emb.shape[1]))
            ids_ds[offset:new_size] = ids_arr
            emb_ds[offset:new_size, :] = emb
            offset = new_size
    print(f"Shard {shard_idx}: wrote {offset} doc embeddings to {out_path} (start={start_line}, count={max_lines})")

def encode_queries(tsv_path: Path, out_path: Path):
    ids, texts = [], []
    with tsv_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) != 2:
                continue
            ids.append(parts[0]); texts.append(parts[1])
    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32)
    with h5py.File(out_path, "w") as h5f:
        h5f.create_dataset("id", data=np.array(ids, dtype=object), dtype=str_dtype, compression="gzip")
        h5f.create_dataset("embedding", data=emb, compression="gzip")
    print(f"Wrote {len(ids)} query embeddings to {out_path}")

# Encode a contiguous shard of docs
DOC_SHARD_OUT = OUT_DIR / f"doc_embeddings_part{SHARD_IDX}.h5"
encode_docs_shard(DOCS_TSV, DOC_SHARD_OUT, SHARD_IDX, NUM_SHARDS)

# Encode queries once (reuse across shards)
QUERY_OUT = OUT_DIR / "query_embeddings.h5"
if not QUERY_OUT.exists():
    encode_queries(QUERIES_TSV, QUERY_OUT)
else:
    print(f"Query embeddings already exist at {QUERY_OUT}, skipping.")


## After running each shard
- Download `/kaggle/working/artifacts/hnsw_embeddings/doc_embeddings_part{SHARD_IDX}.h5` for each shard and `query_embeddings.h5` (once).
- Combine shards locally or iterate over them to compute dense scores and build tiered HNSW indexes.